# Baseline 3 — detector trained with copy-paste augmentation

Trains a Faster R-CNN on NODE21 plus synthetic positives made by pasting real nodule
patches into clean chests, then evaluates it with the same FROC code as Baseline 1.

Six to ten GPU-hours, against a 12-hour session cap. Everything below assumes the session
will die at least once.

## Two things to settle before running

> **The training recipe must match Baseline 1 or the comparison means nothing.**
> Baseline 1 is `epoch_5` of a run whose hyperparameters nobody wrote down. The defaults
> below are reasonable for torchvision detection, but *reasonable* is not *the same*. If a
> difference in FROC could be caused by a different learning rate, the experiment has not
> isolated the augmentation. Get the original training script or README first.
>
> **There is no recorded train/validation split.** Without it you cannot train on what
> Baseline 1 trained on, or hold out what it held out. The notebook writes a deterministic
> patient-level split so the run is at least self-consistent and repeatable, but a
> comparison against Baseline 1 stays invalid until the real split turns up.

## Also worth deciding: whether to run this at all

F7 means the RadEdit augmentation arm trained on the invalidated images, so a fair
generator comparison needs **both** arms retrained — 12 to 20 GPU-hours, not 6 to 10. And
the paper's centre of gravity has moved to what the generator produces rather than whether
augmenting with it helps. This may be a line in Limitations rather than two training runs.

## 1 · Mount, paths, checkpoint plumbing

In [ ]:
!pip -q install SimpleITK

import os, json, time, shutil, subprocess, random
from pathlib import Path
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'mount failed'

NODE21  = Path('/content/drive/MyDrive/Algoverse/data/node21')
MHA_SRC = NODE21/'images'
ANN_CSV = NODE21/'metadata.csv'

OUT  = Path('/content/baseline3')
DEST = Path('/content/drive/MyDrive/Algoverse/results/baseline3')
for d in [OUT/'aug', OUT/'ckpt', OUT/'dets']:  d.mkdir(parents=True, exist_ok=True)
for d in [DEST/'ckpt', DEST/'dets']:           d.mkdir(parents=True, exist_ok=True)
assert MHA_SRC.exists() and ANN_CSV.exists()


def to_drive(sub):
    n = 0
    for f in (OUT/sub).iterdir():
        if not (DEST/sub/f.name).exists():
            shutil.copy(f, DEST/sub/f.name); n += 1
    return n

def from_drive(sub):
    n = 0
    for f in (DEST/sub).iterdir():
        if not (OUT/sub/f.name).exists():
            shutil.copy(f, OUT/sub/f.name); n += 1
    return n

print(f'restored {from_drive("ckpt")} checkpoints, {from_drive("dets")} detection files')

## 2 · Configuration

`EPOCHS`, `LR` and the rest are the numbers that must match Baseline 1. They are grouped
here so they can be replaced in one place once the original recipe is found.

In [ ]:
SIZE, DET_SIZE, SCORE_MIN = 512, 800, 0.05
SEED = 0

# ---- must match Baseline 1 -------------------------------------------------
EPOCHS      = 5          # Baseline 1's archive root is 'epoch_5'
LR          = 0.005
MOMENTUM    = 0.9
WEIGHT_DECAY= 0.0005
LR_STEP     = 3
LR_GAMMA    = 0.1
BATCH       = 4
# ----------------------------------------------------------------------------

AUG_PER_CHEST = 2        # synthetic positives per clean background
N_AUG_CHESTS  = 600      # clean chests to paste into
VAL_FRAC      = 0.2

CHECKPOINT_EVERY_EPOCH = True     # non-negotiable at 6-10 h against a 12 h cap

random.seed(SEED)
import numpy as np, torch
np.random.seed(SEED); torch.manual_seed(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEV == 'cuda', 'training on CPU is not viable here'
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

## 3 · Split

Written to Drive on first run and reloaded thereafter, so every later stage — and any
rerun after a dead session — uses the same one.

Split is by **image**, which is the best available proxy for patient here. NODE21 does not
ship patient identifiers, so two studies of the same patient could land on opposite sides.
That is a limitation to state, not one this notebook can fix.

In [ ]:
import pandas as pd

SPLIT = DEST/'split.json'
raw = pd.read_csv(ANN_CSV)
raw = raw[raw.img_name != 'n0507.mha']          # duplicate of n1059
pos = sorted(raw[raw.label == 1].img_name.unique())
neg = sorted(set(raw[raw.label == 0].img_name) - set(pos))

if SPLIT.exists():
    sp = json.load(open(SPLIT))
    print(f'loaded existing split: {len(sp["train"])} train / {len(sp["val"])} val')
else:
    rng = np.random.default_rng(SEED)
    def cut(xs):
        xs = list(xs); rng.shuffle(xs)
        k = int(len(xs)*(1-VAL_FRAC)); return xs[:k], xs[k:]
    ptr, pva = cut(pos); ntr, nva = cut(neg)
    sp = {'train': sorted(ptr+ntr), 'val': sorted(pva+nva),
          'train_pos': sorted(ptr), 'val_pos': sorted(pva), 'seed': SEED}
    json.dump(sp, open(SPLIT,'w'), indent=1)
    print(f'created split: {len(sp["train"])} train / {len(sp["val"])} val '
          f'({len(pva)} val positives)')

TRAIN, VAL = set(sp['train']), set(sp['val'])
assert not (TRAIN & VAL), 'train and val overlap'
print('WARNING: this is not Baseline 1\'s split. Comparisons against it are invalid '
      'until the original is recovered.')

## 4 · Stage source images

Bulk copy, because reading `.mha` one at a time over the mount stalls inside a C call
where `KeyboardInterrupt` cannot reach it.

In [ ]:
LOCAL = Path('/content/node21'); LOCAL.mkdir(exist_ok=True)
need = sorted(TRAIN | VAL)
t0, staged = time.time(), 0
for i, n in enumerate(need):
    if not (LOCAL/n).exists():
        subprocess.run(['cp', str(MHA_SRC/n), str(LOCAL/n)], check=True); staged += 1
    if i % 400 == 0: print(f'  {i}/{len(need)}  ({time.time()-t0:.0f}s)')
MHA_DIR = LOCAL
print(f'staged {staged} new, {len(list(LOCAL.glob("*.mha")))} present, '
      f'{sum(f.stat().st_size for f in LOCAL.glob("*.mha"))/1e9:.1f} GB, '
      f'{time.time()-t0:.0f}s')

## 5 · Preprocessing and the paste

In [ ]:
import cv2, SimpleITK as sitk
from PIL import Image

def load_chest(path, size=SIZE):
    a = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
    a = a.squeeze() if a.ndim == 3 else a
    oh, ow = a.shape
    s = min(oh, ow); x0, y0 = (ow-s)//2, (oh-s)//2
    a = a[y0:y0+s, x0:x0+s]
    lo, hi = np.percentile(a, [1, 99])
    a = np.clip((a-lo)/(hi-lo+1e-8), 0, 1)
    a = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)) \
           .apply((a*255).astype(np.uint8)).astype(np.float32)/255.0
    a = np.clip(cv2.resize(a, (size,size), interpolation=cv2.INTER_AREA), 0, 1)
    return a, (ow, oh, s, x0, y0)


def feather(n, f=0.25):
    y, x = np.mgrid[0:n, 0:n]
    r = np.sqrt((x-(n-1)/2)**2 + (y-(n-1)/2)**2)/((n-1)/2)
    a = np.ones_like(r); e = (r > 1-f) & (r <= 1)
    a[e] = 0.5*(1+np.cos(np.pi*(r[e]-(1-f))/f)); a[r > 1] = 0
    return a


def paste(dest, patch, cx, cy, diam):
    """Intensity matched on the SURROUND, not the core.

    Matching the core sets the nodule to the same brightness as the lung it replaces and
    erases the contrast that makes it a nodule -- an earlier version did exactly that and
    produced dark discs. Aligning the surround keeps the lesion's own contrast while
    removing the seam.
    """
    d = int(round(diam))
    px, py = int(round(cx-d/2)), int(round(cy-d/2))
    if px < 0 or py < 0 or px+d > SIZE or py+d > SIZE:
        return None, None
    p = cv2.resize(patch, (d, d), interpolation=cv2.INTER_AREA)
    a = feather(d); reg = dest[py:py+d, px:px+d]
    core, ring = a > 0.9, (a > 0.05) & (a < 0.6)
    if ring.sum() < 8: return None, None
    p = np.clip(p + (reg[ring].mean() - p[ring].mean()), 0, 1)
    out = dest.copy(); out[py:py+d, px:px+d] = a*p + (1-a)*reg
    return out, (px, py, px+d, py+d)

## 6 · Build the augmented training set

CPU only, and resumable — it skips anything already written. Only **training** chests are
used as canvases; pasting into a validation chest would leak.

In [ ]:
AUG_CSV = OUT/'augmented.csv'
if AUG_CSV.exists():
    aug = pd.read_csv(AUG_CSV)
    print(f'{len(aug)} augmented images already generated')
else:
    # nodule patches, from TRAINING positives only
    patches = []
    for name, g in raw[raw.label == 1].groupby('img_name'):
        if name not in TRAIN: continue
        img, _ = load_chest(MHA_DIR/name)
        H, W = img.shape
        for r in g.itertuples():
            side = int(max(r.width, r.height) * SIZE / max(W, H) * (W/SIZE))
            side = int(max(r.width, r.height))
            cx, cy = r.x + r.width/2, r.y + r.height/2
            a, (ow, oh, s, x0, y0) = load_chest(MHA_DIR/name)
            fx, fy = (cx-x0)/s, (cy-y0)/s
            fs = side/s
            if not (0 < fs < 0.3 and 0.05 < fx < 0.95 and 0.05 < fy < 0.95): continue
            d = int(fs*SIZE)
            if d < 20: continue
            px, py = int(fx*SIZE-d/2), int(fy*SIZE-d/2)
            if px < 0 or py < 0 or px+d > SIZE or py+d > SIZE: continue
            patches.append(a[py:py+d, px:px+d].copy())
    print(f'{len(patches)} nodule patches from training images')
    assert patches, 'no usable patches'

    clean = [n for n in sorted(set(raw[raw.label == 0].img_name) & TRAIN)][:N_AUG_CHESTS]
    rng = np.random.default_rng(SEED)
    rows = []
    for i, name in enumerate(clean):
        stem = name.replace('.mha','')
        bg, _ = load_chest(MHA_DIR/name)
        for k in range(AUG_PER_CHEST):
            iid = f'{stem}_aug{k}'
            if (OUT/'aug'/f'{iid}.png').exists(): continue
            p = patches[rng.integers(len(patches))]
            # anywhere in the middle two-thirds -- crude, but this is an augmentation set,
            # not the controlled grid
            cx, cy = rng.uniform(0.2, 0.8)*SIZE, rng.uniform(0.2, 0.8)*SIZE
            d = p.shape[0]
            comp, box = paste(bg, p, cx, cy, d)
            if comp is None: continue
            Image.fromarray((comp*255).astype(np.uint8)).save(OUT/'aug'/f'{iid}.png')
            rows.append(dict(image_id=iid, src=stem, x0=box[0], y0=box[1],
                             x1=box[2], y1=box[3], patch_px=d))
        if i % 100 == 0: print(f'  {i}/{len(clean)}  {len(rows)} written')
    aug = pd.DataFrame(rows)
    aug.to_csv(AUG_CSV, index=False); shutil.copy(AUG_CSV, DEST)
    print(f'{len(aug)} augmented images')

shutil.make_archive('/content/aug', 'zip', OUT/'aug')
shutil.copy('/content/aug.zip', DEST/'augmented_images.zip')
print('augmented set backed up to Drive')

## 7 · Dataset and model

In [ ]:
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF

class CXR(Dataset):
    """Real NODE21 images plus pasted ones. Boxes in absolute 512-space pixels."""
    def __init__(self, real_names, aug_df, gt):
        self.items = [('real', n) for n in real_names] + \
                     [('aug', r.image_id) for r in aug_df.itertuples()]
        self.aug = aug_df.set_index('image_id') if len(aug_df) else aug_df
        self.gt = gt
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        kind, key = self.items[i]
        if kind == 'real':
            a, (ow, oh, s, x0, y0) = load_chest(MHA_DIR/key)
            boxes = []
            for r in self.gt.get(key, pd.DataFrame()).itertuples():
                fx0, fy0 = (r.x-x0)/s, (r.y-y0)/s
                fx1, fy1 = (r.x+r.width-x0)/s, (r.y+r.height-y0)/s
                if 0 <= fx0 < fx1 <= 1 and 0 <= fy0 < fy1 <= 1:
                    boxes.append([fx0*SIZE, fy0*SIZE, fx1*SIZE, fy1*SIZE])
        else:
            a = np.asarray(Image.open(OUT/'aug'/f'{key}.png').convert('L'),
                           np.float32)/255.0
            r = self.aug.loc[key]
            boxes = [[r.x0, r.y0, r.x1, r.y1]]
        img = torch.from_numpy(np.stack([a]*3)).float()
        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        return img, {'boxes': boxes,
                     'labels': torch.ones(len(boxes), dtype=torch.int64)}

def collate(b): return tuple(zip(*b))

def new_model():
    m = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None, weights_backbone='DEFAULT')
    m.roi_heads.box_predictor = FastRCNNPredictor(
        m.roi_heads.box_predictor.cls_score.in_features, 2)
    return m

gt_all = {n: g for n, g in raw[raw.label == 1].groupby('img_name')}
train_ds = CXR(sorted(TRAIN), aug, gt_all)
print(f'{len(train_ds)} training items ({len(TRAIN)} real + {len(aug)} pasted)')

## 8 · Train — checkpointed every epoch

Each epoch writes to Drive before starting the next. A dead session costs one epoch, not
the run. Re-running this cell resumes from the last checkpoint Drive has.

In [ ]:
def latest_ckpt():
    cs = sorted((OUT/'ckpt').glob('epoch_*.pth'),
                key=lambda p: int(p.stem.split('_')[1]))
    return cs[-1] if cs else None

model = new_model().to(DEV)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=LR_STEP, gamma=LR_GAMMA)

start = 0
c = latest_ckpt()
if c:
    st = torch.load(c, map_location=DEV, weights_only=False)
    model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
    sched.load_state_dict(st['sched']); start = st['epoch']
    print(f'resuming from {c.name}, epoch {start}')

loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                    num_workers=2, collate_fn=collate)

for ep in range(start, EPOCHS):
    model.train(); t0, tot = time.time(), 0.0
    for i, (imgs, tgts) in enumerate(loader):
        imgs = [im.to(DEV) for im in imgs]
        tgts = [{k: v.to(DEV) for k, v in t.items()} for t in tgts]
        tgts = [t for t in tgts if len(t['boxes'])]          # skip empty targets
        if len(tgts) != len(imgs):
            keep = [j for j, t in enumerate(tgts)]; imgs = imgs[:len(tgts)]
        if not imgs: continue
        loss = sum(model(imgs, tgts).values())
        opt.zero_grad(); loss.backward(); opt.step()
        tot += float(loss)
        if i % 50 == 0:
            el = time.time()-t0
            print(f'  ep{ep} {i}/{len(loader)}  loss {float(loss):.4f}  '
                  f'{el/60:.1f} min  ~{el/(i+1)*(len(loader)-i-1)/60:.0f} min left')
    sched.step()
    p = OUT/'ckpt'/f'epoch_{ep+1}.pth'
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                'sched': sched.state_dict(), 'epoch': ep+1,
                'mean_loss': tot/max(len(loader),1),
                'config': dict(EPOCHS=EPOCHS, LR=LR, MOMENTUM=MOMENTUM,
                               WEIGHT_DECAY=WEIGHT_DECAY, LR_STEP=LR_STEP,
                               LR_GAMMA=LR_GAMMA, BATCH=BATCH, SEED=SEED,
                               AUG_PER_CHEST=AUG_PER_CHEST,
                               N_AUG_CHESTS=N_AUG_CHESTS)}, p)
    to_drive('ckpt')
    print(f'epoch {ep+1}/{EPOCHS} done, mean loss {tot/max(len(loader),1):.4f} '
          f'-> {p.name} saved to Drive [safe]')
print('training complete')

## 9 · Evaluate — the same FROC code as Baseline 1

Scoring is resumable and syncs to Drive, for the same reason training is.

In [ ]:
model.eval()

@torch.no_grad()
def detect_all(arr01, size=DET_SIZE):
    im = Image.fromarray((arr01*255).astype(np.uint8)).convert('RGB') \
              .resize((size,size), Image.LANCZOS)
    o = model([TF.to_tensor(im).to(DEV)])[0]
    b, s = o['boxes'].cpu().numpy()/size, o['scores'].cpu().numpy()
    k = s >= SCORE_MIN
    return b[k], s[k]

def centre_in(b, t):
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    return t[0] <= cx <= t[2] and t[1] <= cy <= t[3]

val_names = sorted(VAL)
done = {p.stem for p in (OUT/'dets').glob('*.json')}
todo = [n for n in val_names if n.replace('.mha','') not in done]
print(f'{len(done)} scored, {len(todo)} to go')

t0 = time.time()
for i, name in enumerate(todo):
    a, (ow, oh, s, x0, y0) = load_chest(MHA_DIR/name)
    b, sc = detect_all(a)
    json.dump({'boxes': b.tolist(), 'scores': sc.tolist(),
               'crop': [int(ow), int(oh), int(s), int(x0), int(y0)]},
              open(OUT/'dets'/f'{name.replace(".mha","")}.json','w'))
    if (i+1) % 50 == 0 or i == len(todo)-1:
        print(f'  {i+1}/{len(todo)}  {(time.time()-t0)/60:.1f} min  '
              f'+{to_drive("dets")} to Drive')

rows, nod = [], []
for stem in sorted({p.stem for p in (OUT/'dets').glob('*.json')}):
    d = json.load(open(OUT/'dets'/f'{stem}.json'))
    b, sc = np.array(d['boxes']).reshape(-1,4), np.array(d['scores'])
    ow, oh, s, x0, y0 = d['crop']
    for j, r in enumerate(gt_all.get(f'{stem}.mha', pd.DataFrame()).itertuples()):
        fx0, fy0 = (r.x-x0)/s, (r.y-y0)/s
        fx1, fy1 = (r.x+r.width-x0)/s, (r.y+r.height-y0)/s
        if not (0 <= fx0 < fx1 <= 1 and 0 <= fy0 < fy1 <= 1): continue
        t = (fx0, fy0, fx1, fy1)
        hits = [ss for bb, ss in zip(b, sc) if centre_in(bb, t)]
        nod.append(dict(img_name=stem, nodule=j, fx0=fx0, fy0=fy0, fx1=fx1, fy1=fy1,
                        best_score=round(float(max(hits, default=0.0)), 4)))
    rows.append(dict(img_name=stem, n_boxes=int(len(sc)),
                     max_score=round(float(sc.max()) if len(sc) else 0.0, 4)))

img_df, nod_df = pd.DataFrame(rows), pd.DataFrame(nod)
assert len(nod_df) > 0, 'no nodules scored on the validation split'
for df, n in [(img_df,'b3_per_image.csv'), (nod_df,'b3_per_nodule.csv')]:
    df.to_csv(OUT/n, index=False); shutil.copy(OUT/n, DEST/n)
print(f'\n{len(img_df)} val images, {len(nod_df)} nodules')
print(nod_df.best_score.describe().round(3).to_string())

## 10 · FROC and comparison

In [ ]:
def froc(nod_df, img_df, detdir):
    ths = np.unique(np.concatenate([np.linspace(0,1,201), nod_df.best_score.values]))
    cache = []
    for _, r in img_df.iterrows():
        d = json.load(open(detdir/f'{r.img_name}.json'))
        b, sc = np.array(d['boxes']).reshape(-1,4), np.array(d['scores'])
        g = nod_df[nod_df.img_name == r.img_name]
        cache.append((b, sc, [(x.fx0,x.fy0,x.fx1,x.fy1) for x in g.itertuples()]))
    n_img, n_nod, pts = len(img_df), len(nod_df), []
    for t in ths:
        tp = int((nod_df.best_score >= t).sum())
        fp = sum(sum(1 for bb in b[sc>=t] if not any(centre_in(bb,g) for g in tg))
                 for b, sc, tg in cache)
        pts.append((t, tp/n_nod, fp/n_img))
    return pd.DataFrame(pts, columns=['threshold','sensitivity','fp_per_image'])

F3 = froc(nod_df, img_df, OUT/'dets')
F3.to_csv(OUT/'b3_froc.csv', index=False); shutil.copy(OUT/'b3_froc.csv', DEST)
OPS = [0.125,0.25,0.5,1,2,4,8]
T3 = pd.DataFrame([dict(fp_per_image=op,
        sensitivity=round(F3[F3.fp_per_image<=op].sensitivity.max(),4)
        if (F3.fp_per_image<=op).any() else np.nan) for op in OPS])
print('Baseline 3 (copy-paste augmented):'); print(T3.to_string(index=False))
print(f'FROC score: {T3.sensitivity.mean():.4f}')

b1 = Path('/content/drive/MyDrive/Algoverse/results/baselines/table-froc.csv')
if b1.exists():
    T1 = pd.read_csv(b1)
    cmp = T1.merge(T3, on='fp_per_image', suffixes=('_b1','_b3'))
    cmp['delta'] = (cmp.sensitivity_b3 - cmp.sensitivity_b1).round(4)
    print('\nvs Baseline 1:'); print(cmp.to_string(index=False))
    print(f'\nmean FROC: B1 {T1.sensitivity.mean():.4f}  B3 {T3.sensitivity.mean():.4f}  '
          f'delta {T3.sensitivity.mean()-T1.sensitivity.mean():+.4f}')
    print('\nDO NOT report this delta as an augmentation effect unless the training')
    print('recipe and split matched Baseline 1. Otherwise it also contains whatever')
    print('difference those introduce.')
else:
    print('\nBaseline 1 table not found -- run the baselines notebook first.')

T3.to_csv(OUT/'table-b3-froc.csv', index=False); shutil.copy(OUT/'table-b3-froc.csv', DEST)
print(f'\nsynced: {to_drive("ckpt")} checkpoints, {to_drive("dets")} detections')